<a href="https://colab.research.google.com/github/Aditya-0I/micrograd/blob/main/micrograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [110]:
import random
import math
import numpy as np
import pandas as pd

In [111]:
class node:
  def __init__(self,val,prev=(),op=''):
    self.val = val
    self.grad = 0.0
    self.prev = prev
    self.op=op
    self.label = ''
    self.backward = lambda : None

  def __repr__(self):
    return f"{self.val},grad:{self.grad}"

  def __add__(self,other):
    other = other if isinstance(other, node) else node(other)
    out = node(self.val+other.val,(self,other),'+')

    def back():
      self.grad += out.grad
      other.grad += out.grad
    out.backward=back

    return out

  def __mul__(self,other):
    other = other if isinstance(other, node) else node(other)
    out = node(self.val*other.val,(self,other),'*')

    def back():
      self.grad += out.grad * other.val
      other.grad += out.grad * self.val
    out.backward=back

    return out

  def __sub__(self,other):
    other = other if isinstance(other, node) else node(other)
    out = node(self.val-other.val,(self,other),'-')

    def back():
      self.grad += out.grad
      other.grad -= out.grad
    out.backward=back

    return out

  def ReLU(self):
    out = node(max(0,self.val),(self,),'ReLU')

    def back():
      self.grad += out.grad*(out.val>0)
    out.backward=back

    return out

  def PReLU(self):
    out = node(self.val if (self.val>0) else 0.1*self.val,(self,),'ReLU')

    def back():
      self.grad += out.grad*(1 if out.val>0 else 0.1)
    out.backward=back

    return out

  def tanh(self):
    t = math.tanh(self.val)
    out = node(t,(self,),'tanh')

    def back():
      self.grad += out.grad*(1-t*t)
    out.backward = back

    return out


  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = node(self.val**other, (self,), f'**{other}')

    def back():
        self.grad += other * (self.val ** (other - 1)) * out.grad
    out.backward = back

    return out

  def backprop(self):

    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v.prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)

    self.grad = 1.0
    for node in reversed(topo):
      node.backward()


In [112]:
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v.prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.val, n.grad), shape='record')
    if n.op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n.op, label = n.op)
      # and connect this node to it
      dot.edge(uid + n.op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2.op)

  return dot


In [113]:
class Neuron:
  def __init__(self,nin):
    self.w = [node(random.uniform(-1,1)) for i in range(nin)]
    self.b = node(random.uniform(-1,1))

  def __call__(self,x):
    act = sum((wi*ai for wi,ai in zip(self.w,x)),self.b)
    out = act.tanh()
    return out

  def para(self):
    return self.w + [self.b]


class Layer:
  def __init__(self,nin,nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]

  def __call__(self,x):
    out = [n(x) for n in self.neurons]
    return out[0] if len(out)==1 else out

  def para(self):
    return [p for n in self.neurons for p in n.para()]


class MLP:
  def __init__(self,nin:int,nout:list):
    layer_size = [nin] + nout
    self.layers = [Layer(layer_size[i],layer_size[i+1]) for i in range(len(nout))]

  def __call__(self,x):
    for layer in self.layers:
      x = layer(x)
    return x

  def para(self):
    return [p for l in self.layers for p in l.para()]

In [114]:
nn = MLP(3, [9,2])
x = [2.0, 3.0, -1.0]
nn(x)

[0.8362514140753227,grad:0.0, 0.09697550143145975,grad:0.0]

In [115]:
xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = ([1.0, -1.0],[-1.0, 1.0],[-1.0, -1.0],[1.0, -1.0])
for l in ys :
  for y in l:
    y = node(y)
    y.label='input'

In [116]:
for k in range(500):
  ypred = [nn(x) for x in xs]
  loss = sum(((yp-y)**2 for lpred,l in zip(ypred,ys) for yp,y in zip(lpred,l)),node(0))
  loss.label='loss'


  for p in nn.para():
    p.grad=0.0

  loss.backprop()

  for p in nn.para():
    p.val += -0.1*p.grad


  if (k%10 == 0): print(k,loss.val)



0 13.73276940161109
10 0.03405045325198216
20 0.02266789668449908
30 0.017315829785486465
40 0.013988225539627178
50 0.01170150318426346
60 0.01003288751588389
70 0.008763155924898876
80 0.007766021103223255
90 0.00696333960150464
100 0.006304116062107878
110 0.005753655078080899
120 0.005287539875461793
130 0.004888098108514311
140 0.004542231468588728
150 0.004240031752473854
160 0.0039738698305820455
170 0.0037377792419189654
180 0.003527029044608131
190 0.0033378215200868663
200 0.0031670742019679673
210 0.003012260058249075
220 0.0028712885349880175
230 0.0027424157991718426
240 0.0026241761681593894
250 0.002515329127227363
260 0.002414817963092132
270 0.0023217371553165746
280 0.0022353064423730027
290 0.002154850025714101
300 0.0020797797657987247
310 0.0020095815065218313
320 0.001943803871121545
330 0.0018820490253434459
340 0.0018239650176081232
350 0.0017692393917600761
360 0.001717593833175289
370 0.0016687796589323985
380 0.0016225740012756886
390 0.0015787765635371395
40

In [117]:
ypred

[[0.9900345669709482,grad:-0.01993086605810368,
  -0.998967850016806,grad:0.002064299966388017],
 [-0.9851699507937786,grad:0.02966009841244288,
  0.9862706810961038,grad:-0.02745863780779234],
 [-0.9832042277362811,grad:0.03359154452743773,
  -0.9965265309689733,grad:0.006946938062053487],
 [0.9853701358035334,grad:-0.029259728392933226,
  -0.9858525400982214,grad:0.02829491980355714]]